In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [19]:
# ── 1. Load Data ──────────────────────────────────────────
# IEEE dataset has two files - transactions + identity
# Both must be joined on TransactionID

train_transaction = pd.read_csv('../data/raw/train_transaction.csv')
train_identity    = pd.read_csv('../data/raw/train_identity.csv')

print("Transaction Table Shape:", train_transaction.shape)
print("Identity Table Shape   :", train_identity.shape)

Transaction Table Shape: (590540, 394)
Identity Table Shape   : (144233, 41)


In [20]:
# ── 2. Merge Tables ───────────────────────────────────────
# Left join: keep all transactions, attach identity where available
# Not all transactions have identity data — that itself is a signal

df = pd.merge(train_transaction,train_identity,on='TransactionID',how='left')
print("\nMerged Dataset Shape:", df.shape)


Merged Dataset Shape: (590540, 434)


In [21]:
# ── 3. First Business Look ────────────────────────────────

total_transactions = len(df)
total_fraud        = df['isFraud'].sum()
fraud_rate         = (total_fraud / total_transactions) * 100

total_amount       = df['TransactionAmt'].sum()
fraud_amount       = df[df['isFraud'] == 1]['TransactionAmt'].sum()
fraud_amount_pct   = (fraud_amount / total_amount) * 100

print("\n========== BUSINESS SUMMARY ==========")
print(f"Total Transactions  : {total_transactions:,}")
print(f"Fraudulent Txns     : {total_fraud:,}")
print(f"Fraud Rate          : {fraud_rate:.2f}%")
print(f"Total Amount        : ${total_amount:,.2f}")
print(f"Fraud Amount        : ${fraud_amount:,.2f}")
print(f"Fraud Amount %      : {fraud_amount_pct:.2f}%")
print("=======================================")


========== BUSINESS SUMMARY ==========
Total Transactions  : 590,540
Fraudulent Txns     : 20,663
Fraud Rate          : 3.50%
Total Amount        : $79,738,948.73
Fraud Amount        : $3,083,844.86
Fraud Amount %      : 3.87%


In [22]:
# ── 4. Column Inventory ───────────────────────────────────
print("\n--- Transaction Table Columns ---")
print(f"Total columns: {train_transaction.shape[1]}")
print(train_transaction.dtypes.value_counts())

print("\n--- Identity Table Columns ---")
print(f"Total columns: {train_identity.shape[1]}")
print(train_identity.dtypes.value_counts())


--- Transaction Table Columns ---
Total columns: 394
float64    376
object      14
int64        4
Name: count, dtype: int64

--- Identity Table Columns ---
Total columns: 41
float64    23
object     17
int64       1
Name: count, dtype: int64


In [23]:
# ── 5. Key Column Peek ────────────────────────────────────
key_cols = [
    'TransactionID', 'isFraud', 'TransactionDT','TransactionAmt', 'ProductCD', 'card4', 'card6',
    'P_emaildomain', 'R_emaildomain'
]

print("\n--- Key Columns Sample ---")
print(df[key_cols].head(10).to_string())


--- Key Columns Sample ---
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD       card4   card6  P_emaildomain R_emaildomain
0        2987000        0          86400            68.5         W    discover  credit            NaN           NaN
1        2987001        0          86401            29.0         W  mastercard  credit      gmail.com           NaN
2        2987002        0          86469            59.0         W        visa   debit    outlook.com           NaN
3        2987003        0          86499            50.0         W  mastercard   debit      yahoo.com           NaN
4        2987004        0          86506            50.0         H  mastercard  credit      gmail.com           NaN
5        2987005        0          86510            49.0         W        visa   debit      gmail.com           NaN
6        2987006        0          86522           159.0         W        visa   debit      yahoo.com           NaN
7        2987007        0          86529    

In [24]:
# ── 6. Class Imbalance Check ──────────────────────────────
print("\n--- Class Distribution ---")
print(df['isFraud'].value_counts())
print(df['isFraud'].value_counts(normalize=True).mul(100).round(2))


--- Class Distribution ---
isFraud
0    569877
1     20663
Name: count, dtype: int64
isFraud
0    96.5
1     3.5
Name: proportion, dtype: float64


In [25]:
# ── 7. Missing Value Overview ─────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'Missing Count': missing,'Missing %': missing_pct}).sort_values('Missing %', ascending=False)

# Only show columns with actual missing values
missing_report = missing_report[missing_report['Missing Count'] > 0]

print(f"\n--- Missing Value Report ---")
print(f"Columns with missing values: {len(missing_report)}")
print(missing_report.head(20).to_string())


--- Missing Value Report ---
Columns with missing values: 414
       Missing Count  Missing %
id_24         585793      99.20
id_25         585408      99.13
id_26         585377      99.13
id_21         585381      99.13
id_08         585385      99.13
id_07         585385      99.13
id_27         585371      99.12
id_23         585371      99.12
id_22         585371      99.12
dist2         552913      93.63
D7            551623      93.41
id_18         545427      92.36
D13           528588      89.51
D14           528353      89.47
D12           525823      89.04
id_03         524216      88.77
id_04         524216      88.77
D6            517353      87.61
id_33         517251      87.59
id_09         515614      87.31


In [26]:
# ── 8. Transaction Amount Distribution ───────────────────
print("\n--- Transaction Amount Stats ---")
print(df.groupby('isFraud')['TransactionAmt'].describe().round(2).to_string())


--- Transaction Amount Stats ---
            count    mean     std   min    25%   50%    75%       max
isFraud                                                              
0        569877.0  134.51  239.40  0.25  43.97  68.5  120.0  31937.39
1         20663.0  149.24  232.21  0.29  35.04  75.0  161.0   5191.00


In [28]:
# ── 9. Save Merged Data ───────────────────────────────────
df.to_csv('../data/processed/merged_data.csv', index=False)
print("\nMerged data saved to data/processed/merged_data.csv")


Merged data saved to data/processed/merged_data.csv
